In [1]:
import os

os.environ['PYSPARK_SUBMIT_ARGS'] = \
'--packages io.delta:delta-core_2.12:1.0.0 pyspark-shell'

In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

spark = SparkSession. \
    builder. \
    master('yarn'). \
    config('spark.ui.port', '0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension"). \
    config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    getOrCreate()

In [3]:
spark.conf.get("spark.sql.extensions")

'io.delta.sql.DeltaSparkSessionExtension'

### Task 1 Part 1 (Schema Evaluation)

In [4]:
df_day1 = spark.createDataFrame([
    (1, "Ankit", "Mumbai"),
    (2, "Rahul", "Pune"),
    (3, "Amit", "Delhi"),
    (4, "Neha", "Bangalore"),
    (5, "Kiran", "Hyderabad"),
    (6, "Sneha", "Chennai"),
    (7, "Rohit", "Nagpur"),
    (8, "Priya", "Kolkata"),
    (9, "Vikas", "Jaipur"),
    (10, "Pooja", "Ahmedabad")
], ["id", "name", "city"])

In [5]:
df_day1.write \
.format("delta") \
.save("/user/itv020178/delta_format")

In [6]:
df_day2 = spark.createDataFrame([
    (11, "Arjun", "Mumbai", 28),
    (12, "Meena", "Pune", 32),
    (13, "Suresh", "Delhi", 40),
    (14, "Kavita", "Bangalore", 27),
    (15, "Ramesh", "Hyderabad", 35),
    (16, "Anjali", "Chennai", 29),
    (17, "Deepak", "Nagpur", 31),
    (18, "Sunita", "Kolkata", 45),
    (19, "Manoj", "Jaipur", 38),
    (20, "Rekha", "Ahmedabad", 33)
], ["id", "name", "city", "age"])

In [10]:
df_day2.write \
.format("delta") \
.mode("append") \
.option("mergeSchema","true") \
.save("/user/itv020178/delta_format")


In [11]:
df_day3 = spark.createDataFrame([
    (21, "Tarun", "Mumbai", 26, "tarun@gmail.com"),
    (22, "Nisha", "Pune", 30, "nisha@gmail.com"),
    (23, "Vivek", "Delhi", 36, "vivek@gmail.com"),
    (24, "Swati", "Bangalore", 28, "swati@gmail.com"),
    (25, "Alok", "Hyderabad", 34, "alok@gmail.com"),
    (26, "Divya", "Chennai", 27, "divya@gmail.com"),
    (27, "Harsh", "Nagpur", 29, "harsh@gmail.com"),
    (28, "Komal", "Kolkata", 31, "komal@gmail.com"),
    (29, "Nitin", "Jaipur", 37, "nitin@gmail.com"),
    (30, "Ritu", "Ahmedabad", 35, "ritu@gmail.com")
], ["id", "name", "city", "age", "email"])

In [12]:
df_day3.write \
.format("delta") \
.option("mergeSchema","true") \
.mode("append") \
.save("/user/itv020178/delta_format")

# Task 2 Part 1 (Time Travel)

In [13]:
df_v0 = spark.read.format("delta") \
.option("versionAsOf", 0) \
.load("/user/itv020178/delta_format")

In [14]:
df_v0.show()

+---+-----+---------+
| id| name|     city|
+---+-----+---------+
|  6|Sneha|  Chennai|
|  7|Rohit|   Nagpur|
|  8|Priya|  Kolkata|
|  9|Vikas|   Jaipur|
| 10|Pooja|Ahmedabad|
|  1|Ankit|   Mumbai|
|  2|Rahul|     Pune|
|  3| Amit|    Delhi|
|  4| Neha|Bangalore|
|  5|Kiran|Hyderabad|
+---+-----+---------+



In [15]:
df_v1 = spark.read.format("delta") \
.option("versionAsOf", 1) \
.load("/user/itv020178/delta_format")

In [16]:
df_v1.show()

+---+------+---------+----+
| id|  name|     city| age|
+---+------+---------+----+
| 17|Deepak|   Nagpur|  31|
| 18|Sunita|  Kolkata|  45|
| 19| Manoj|   Jaipur|  38|
| 20| Rekha|Ahmedabad|  33|
| 14|Kavita|Bangalore|  27|
| 15|Ramesh|Hyderabad|  35|
| 16|Anjali|  Chennai|  29|
| 11| Arjun|   Mumbai|  28|
| 12| Meena|     Pune|  32|
| 13|Suresh|    Delhi|  40|
|  6| Sneha|  Chennai|null|
|  7| Rohit|   Nagpur|null|
|  8| Priya|  Kolkata|null|
|  9| Vikas|   Jaipur|null|
| 10| Pooja|Ahmedabad|null|
|  1| Ankit|   Mumbai|null|
|  2| Rahul|     Pune|null|
|  3|  Amit|    Delhi|null|
|  4|  Neha|Bangalore|null|
|  5| Kiran|Hyderabad|null|
+---+------+---------+----+



In [17]:
from delta.tables import DeltaTable

In [18]:
delta_table = DeltaTable.forPath(spark, "/user/itv020178/delta_format")

In [19]:
delta_table.history().show()

+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+
|version|           timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|
+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+
|      2|2026-04-29 06:48:...|  null|    null|    WRITE|{mode -> Append, ...|null|    null|     null|          1|          null|         true|{numFiles -> 2, n...|        null|
|      1|2026-04-29 06:45:...|  null|    null|    WRITE|{mode -> Append, ...|null|    null|     null|          0|          null|         true|{numFiles -> 3, n...|        null|
|      0|2026-04-29 06:41:...|  null|    null|    WRITE|{mode -> ErrorIfE...|null|    null|     null|       null|  

In [20]:
df_latest = spark.read.format("delta") \
.load("/user/itv020178/delta_format")

In [21]:
df_latest.show()

+---+------+---------+---+---------------+
| id|  name|     city|age|          email|
+---+------+---------+---+---------------+
| 26| Divya|  Chennai| 27|divya@gmail.com|
| 27| Harsh|   Nagpur| 29|harsh@gmail.com|
| 28| Komal|  Kolkata| 31|komal@gmail.com|
| 29| Nitin|   Jaipur| 37|nitin@gmail.com|
| 30|  Ritu|Ahmedabad| 35| ritu@gmail.com|
| 21| Tarun|   Mumbai| 26|tarun@gmail.com|
| 22| Nisha|     Pune| 30|nisha@gmail.com|
| 23| Vivek|    Delhi| 36|vivek@gmail.com|
| 24| Swati|Bangalore| 28|swati@gmail.com|
| 25|  Alok|Hyderabad| 34| alok@gmail.com|
| 17|Deepak|   Nagpur| 31|           null|
| 18|Sunita|  Kolkata| 45|           null|
| 19| Manoj|   Jaipur| 38|           null|
| 20| Rekha|Ahmedabad| 33|           null|
| 14|Kavita|Bangalore| 27|           null|
| 15|Ramesh|Hyderabad| 35|           null|
| 16|Anjali|  Chennai| 29|           null|
| 11| Arjun|   Mumbai| 28|           null|
| 12| Meena|     Pune| 32|           null|
| 13|Suresh|    Delhi| 40|           null|
+---+------